# DS2002 · SQL Challenge Set

**Lab — 2026-09-11 · Fall 2026**  

---

## Lab 03 — SQL Challenge Set

Seven questions, one query each. Every query has to produce the right answer when the notebook is run from a fresh kernel, top to bottom.

Two rules that matter as much as getting the answer:

- **Check the row count** against what you expect before you believe a result.
- **Decide what to do about the untagged track and the unplayed tracks.** Several of these questions have a defensible answer either way; what is not defensible is not noticing they exist.

In [72]:
import sqlite3, pandas as pd
conn = sqlite3.connect(':memory:')
cur = conn.cursor()
cur.executescript('''
CREATE TABLE artists (artist_id INTEGER PRIMARY KEY, name TEXT, country TEXT);
CREATE TABLE tracks (track_id INTEGER PRIMARY KEY, title TEXT, artist_id INTEGER, genre TEXT, seconds INTEGER);
CREATE TABLE plays (play_id INTEGER PRIMARY KEY, track_id INTEGER, user TEXT, played_on TEXT);
INSERT INTO artists VALUES
 (1,'Nova Waves','US'),(2,'The Blue Ridge','US'),(3,'Kestrel','UK'),(4,'Marisol','ES');
INSERT INTO tracks VALUES
 (10,'Skyline',1,'Pop',201),(11,'Undertow',1,'Pop',240),(12,'Foothills',2,'Folk',185),
 (13,'Aurora',3,'Electronic',300),(14,'Nightfall',3,'Electronic',275),(15,'Sol',4,'Latin',210),
 (16,'Coastline',2,'Folk',199),(17,'Ridgeline',2,'Folk',225),(18,'Untitled Demo',3,NULL,150);
INSERT INTO plays VALUES
 (100,10,'ava','2026-09-01'),(101,10,'ben','2026-09-01'),(102,13,'ava','2026-09-02'),
 (103,13,'cara','2026-09-02'),(104,14,'ben','2026-09-03'),(105,12,'ava','2026-09-03'),
 (106,15,'dan','2026-09-04'),(107,10,'cara','2026-09-04'),(108,13,'dan','2026-09-05'),
 (109,16,'ava','2026-09-05'),(110,11,'ben','2026-09-06');
''')
conn.commit()

def q(sql):
    return pd.read_sql_query(sql, conn)
print('ready')

ready


### Q1 — Every track with its artist's name and country.

*Expected: 9 rows, one per track.*

In [73]:
q1 = q('''

SELECT t.title, a.name, a.country
FROM tracks AS t
JOIN artists AS a
    ON t.artist_id = a.artist_id;

''')

q1

,title,name,country
0,Skyline,Nova Waves,US
1,Undertow,Nova Waves,US
2,Foothills,The Blue Ridge,US
3,Aurora,Kestrel,UK
4,Nightfall,Kestrel,UK
5,Sol,Marisol,ES
6,Coastline,The Blue Ridge,US
7,Ridgeline,The Blue Ridge,US
8,Untitled Demo,Kestrel,UK


I chose the track title from the tracks table and the artist name and country from the artists table. I joined the two tables using artist_id so SQL could match each track to the correct artist and return 9 tracks with their artist information.

### Q2 — Which genre has the longest average track length?

Return the genre and the average, not just the name.

In [74]:
q2 = q('''

SELECT genre, AVG(seconds) AS avg_length
FROM tracks
WHERE genre IS NOT NULL
GROUP BY genre
ORDER BY avg_length DESC
LIMIT 1;

''')

q2

,genre,avg_length
0,Electronic,287.5


I grouped the tracks by genre and used AVG(seconds) to calculate the average track length for each genre. I left out tracks with a missing genre, sorted the averages from highest to lowest, and used LIMIT 1 to return only the genre with the longest average track length.

### Q3 — For each user: how many plays, and how many distinct tracks?

Someone who played one track four times is a different listener from someone who played four different tracks. Your result should make that visible.

In [75]:
q3 = q('''

SELECT user,
       COUNT(*) AS total_plays,
       COUNT(DISTINCT track_id) AS distinct_tracks
FROM plays
GROUP BY user;

''')

q3

,user,total_plays,distinct_tracks
0,ava,4,4
1,ben,3,3
2,cara,2,2
3,dan,2,2


I grouped the plays by user and used COUNT(*) to find each user’s total number of plays. I used COUNT(DISTINCT track_id) to count how many different tracks each user played, so repeated plays of the same track would only count once toward distinct tracks.

### Q4 — Which tracks have never been played?

*Expected: 2 rows.* Hint: `LEFT JOIN` and then keep the rows where the right side came back `NULL`.

In [76]:
q4 = q('''

SELECT t.track_id, t.title
FROM tracks AS t
LEFT JOIN plays AS p
    ON t.track_id = p.track_id
WHERE p.play_id IS NULL;

''')

q4

,track_id,title
0,17,Ridgeline
1,18,Untitled Demo


I used a LEFT JOIN to keep every track, including tracks that did not have a matching play. I used IS NULL to find the tracks with no matching play ID, which showed the tracks that had never been played.

### Q5 — Rank artists by total listening time.

Sum the seconds actually listened across all plays, most to least, and include a minutes column rounded to one decimal.

In [77]:
q5 = q('''

SELECT a.name,
       SUM(t.seconds) AS total_seconds,
       ROUND(SUM(t.seconds) / 60.0, 1) AS total_minutes
FROM artists AS a
JOIN tracks AS t
    ON a.artist_id = t.artist_id
JOIN plays AS p
    ON t.track_id = p.track_id
GROUP BY a.artist_id, a.name
ORDER BY total_seconds DESC;

''')

q5

,name,total_seconds,total_minutes
0,Kestrel,1175,19.6
1,Nova Waves,843,14.1
2,The Blue Ridge,384,6.4
3,Marisol,210,3.5


I joined the artists, tracks, and plays tables so I could connect each play to the correct artist and track length. I added the track seconds across all plays, changed total to minutes, and rounded it to one decimal. I sorted the artists from the most total listening time to the least.

### Q6 — Which tracks are missing a genre?

Return the track id and title. Then, in a comment, say what `WHERE genre != 'Pop'` would have done to these rows and why.

In [78]:
q6 = q('''

SELECT track_id, title
FROM tracks
WHERE genre IS NULL;

''')

# WHERE genre != 'Pop' would skip this track since SQL can't compare a missing genre to 'Pop'

q6


,track_id,title
0,18,Untitled Demo


I used IS NULL to find tracks where the genre was missing. SQL uses IS NULL for missing values because NULL cannot be compared the same way as a normal genre value.

### Q7 — Plays per day.

`played_on` is stored as text like `'2026-09-01'`. Count plays per date, earliest first, and include the number of distinct users active that day.

In [79]:
q7 = q('''

SELECT played_on,
       COUNT(*) AS total_plays,
       COUNT(DISTINCT user) AS distinct_users
FROM plays
GROUP BY played_on
ORDER BY played_on ASC;

''')

q7

,played_on,total_plays,distinct_users
0,2026-09-01,2,2
1,2026-09-02,2,2
2,2026-09-03,2,2
3,2026-09-04,2,2
4,2026-09-05,2,2
5,2026-09-06,1,1


I grouped the plays by date and used COUNT(*) to find the total number of plays each day. I used COUNT(DISTINCT user) to count how many different users were active each day. After, I sorted the dates from earliest to latest.

### Validate your work

**TODO:** uncomment these and make them pass. Assign your query results to the variables as you go — for example `q1 = q('''...''')`.

In [80]:
assert len(q1) == 9, 'Q1 should return one row per track'
assert len(q4) == 2, 'Q4: two tracks have never been played'
assert q3['total_plays'].sum() == 11, 'Q3 should account for all 11 plays'
print('checks passed.')

checks passed.


### Write-up

Pick the query that gave you the most trouble and explain what you had wrong before you had it right. Name the specific misunderstanding — "I put the aggregate in WHERE" or "I used an inner join and lost the tracks with no plays" — not "it was confusing."

Q5 gave me the most trouble because at first I was thinking I could just add up the length of each artist’s tracks. I realized that the question was asking for actual listening time, so a track’s length needed to be counted every time it was played. Joining the plays table made each play count separately before I summed the seconds for each artist.